# LLM Tooling & Inference — Intern Task Notebook

**Topics covered in this notebook:**

| # | Task | Tools |
|---|------|-------|
| 1 | Install & configure Ollama with Llama 3.1 8B | `ollama` CLI, Python `requests` |
| 2 | Explore Hugging Face Hub | `huggingface_hub`, model cards, licensing |
| 3 | Load & run inference with pre-trained models | `transformers`, `pipeline` |
| 4 | Experiment with generation parameters | `temperature`, `top_p`, `top_k`, `max_tokens` |
| 5 | Build a CLI tool for model inference comparison | `argparse`, Ollama + HF unified interface |

---

> **Before you begin:** Run each cell in order. Markdown cells explain the concept; code cells implement it.  
> Hardware note — Tasks 1–2 can run on any machine. Task 3–4 need at least 8 GB RAM; GPU is optional but speeds things up.

---
## Task 1 — Install & Configure Ollama with Llama 3.1 8B

### What is Ollama?

**Ollama** is a tool that lets you download and run large language models locally on your machine — no internet connection needed after the model is downloaded, no API keys, and no cost per token.

It works like Docker: you `pull` a model image, then `run` it. Behind the scenes it handles quantization, model loading, and serving an HTTP API on `localhost:11434`.

### Why Llama 3.1 8B?

- **8B parameters** — small enough to run on a laptop with 8 GB RAM
- **Meta's open-weights model** — one of the best open-source models at this size
- **Supports long context** — 128k token context window
- **Commercial-friendly license** — can be used in products

### Installation steps (run once in your terminal, not here)

```bash
# Linux / macOS
curl -fsSL https://ollama.com/install.sh | sh

# Windows — download installer from https://ollama.com/download

# After installation, pull the model (downloads ~4.7 GB)
ollama pull llama3.1

# Verify it works
ollama run llama3.1 "Say hello in one sentence."

# List downloaded models
ollama list
```

### How Ollama serves the model

Once installed, Ollama runs a background server at `http://localhost:11434`.  
It exposes two main API endpoints:

| Endpoint | Method | Purpose |
|----------|--------|----------|
| `/api/generate` | POST | Single-turn generation |
| `/api/chat` | POST | Multi-turn chat (messages array) |
| `/api/tags` | GET | List available models |

In [ ]:
# Cell 1.1 — Verify Ollama is running
# This pings the Ollama server and lists available models.

import requests
import json

OLLAMA_BASE = "http://localhost:11434"

try:
    response = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    models = response.json().get("models", [])
    print("Ollama is running!")
    print(f"Models available: {len(models)}")
    for m in models:
        print(f"  - {m['name']}  ({round(m['size'] / 1e9, 2)} GB)")
except requests.exceptions.ConnectionError:
    print("Ollama is NOT running.")
    print("Start it with:  ollama serve  (in a terminal)")
    print("Then pull the model: ollama pull llama3.1")

In [ ]:
# Cell 1.2 — Simple text generation with Ollama API
# We call /api/generate and stream=False to get the full response at once.

def ollama_generate(prompt, model="llama3.1", stream=False, **kwargs):
    """
    Send a prompt to Ollama and return the generated text.

    Parameters
    ----------
    prompt  : str   — The input text
    model   : str   — Model name as shown in 'ollama list'
    stream  : bool  — If True, yields tokens as they arrive
    **kwargs        — Extra generation options (temperature, top_p, etc.)
    """
    payload = {
        "model":  model,
        "prompt": prompt,
        "stream": stream,
        "options": kwargs
    }
    resp = requests.post(f"{OLLAMA_BASE}/api/generate", json=payload)
    resp.raise_for_status()
    return resp.json()["response"]


# Try it
result = ollama_generate(
    prompt="Explain what a large language model is in 2 sentences.",
    model="llama3.1"
)
print(result)

In [ ]:
# Cell 1.3 — Multi-turn chat with Ollama
# /api/chat accepts a messages list (role: user/assistant/system)

def ollama_chat(messages, model="llama3.1", **kwargs):
    """
    Multi-turn chat with Ollama.

    messages : list of dicts — [{"role": "user", "content": "..."}]
    """
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": kwargs
    }
    resp = requests.post(f"{OLLAMA_BASE}/api/chat", json=payload)
    resp.raise_for_status()
    return resp.json()["message"]["content"]


# Conversation with a system persona
messages = [
    {"role": "system",    "content": "You are a helpful Python tutor. Keep answers brief."},
    {"role": "user",      "content": "What is a decorator in Python?"},
]

reply = ollama_chat(messages, model="llama3.1")
print("Assistant:", reply)

# Add the reply to history and continue the conversation
messages.append({"role": "assistant", "content": reply})
messages.append({"role": "user",      "content": "Show me a one-line example."})

reply2 = ollama_chat(messages, model="llama3.1")
print("\nAssistant (turn 2):", reply2)

---
## Task 2 — Explore Hugging Face Hub

### What is Hugging Face Hub?

Hugging Face Hub is the GitHub of AI — a platform hosting over **700,000 models**, **150,000 datasets**, and **300,000 demo apps (Spaces)**.

Key things to know:

| Concept | Description |
|---------|-------------|
| **Model card** | README.md describing the model — architecture, training data, intended use, limitations |
| **Pipeline tag** | Task type: `text-generation`, `fill-mask`, `token-classification`, etc. |
| **License** | Legal terms — ranges from fully open (MIT, Apache 2.0) to restricted (Llama license) |
| **Downloads** | Monthly download count — a proxy for community trust |
| **Safetensors** | Safer weight format vs `.bin` — prefer models with safetensors support |

### Common licenses you'll encounter

| License | Commercial use? | Modification? | Notes |
|---------|----------------|---------------|-------|
| `MIT` | Yes | Yes | Most permissive |
| `Apache 2.0` | Yes | Yes | Must include license notice |
| `CC-BY 4.0` | Yes | Yes | Attribution required |
| `llama3` | Conditional | Yes | Requires Meta agreement; >700M MAU needs special license |
| `gpl-3.0` | Yes | Yes | Derivatives must also be GPL |
| `other` / custom | Check individually | — | Read the model card carefully |

### Installation

```bash
pip install huggingface_hub
```

In [ ]:
# Cell 2.1 — Install Hugging Face Hub library
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "-q"])

In [ ]:
# Cell 2.2 — Search models on Hugging Face Hub programmatically
from huggingface_hub import HfApi, ModelCard

api = HfApi()

# Search top text-generation models sorted by downloads
models = list(api.list_models(
    task="text-generation",
    sort="downloads",
    direction=-1,           # descending
    limit=10
))

print(f"{'Model ID':<45} {'Downloads':>12} {'License':<20}")
print("-" * 80)
for m in models:
    license_tag = ""
    if m.tags:
        for tag in m.tags:
            if tag.startswith("license:"):
                license_tag = tag.replace("license:", "")
                break
    downloads = getattr(m, 'downloads', 0) or 0
    print(f"{m.modelId:<45} {downloads:>12,} {license_tag:<20}")

In [ ]:
# Cell 2.3 — Read a model card
# Model cards contain crucial information: intended use, training data,
# limitations, bias evaluations, and licensing details.

from huggingface_hub import ModelCard

MODEL_ID = "distilbert-base-uncased"   # small model, downloads fast

card = ModelCard.load(MODEL_ID)

# Print first 1500 characters of the model card
print(f"=== Model Card: {MODEL_ID} ===")
print(card.text[:1500])
print("...")

In [ ]:
# Cell 2.4 — Extract structured metadata from a model card
# The YAML front-matter in a model card gives structured info.

from huggingface_hub import model_info

info = model_info(MODEL_ID)

print(f"Model ID     : {info.modelId}")
print(f"Author       : {info.author}")
print(f"Downloads/mo : {getattr(info, 'downloads', 'N/A'):,}")
print(f"Likes        : {info.likes}")
print(f"Library      : {info.library_name}")
print(f"Pipeline tag : {info.pipeline_tag}")
print(f"Tags         : {info.tags[:8]}")
print(f"Private      : {info.private}")
print(f"Created at   : {info.created_at}")

# Siblings = files inside the model repo
print(f"\nFiles in repo:")
for f in info.siblings:
    print(f"  {f.rfilename}")

In [ ]:
# Cell 2.5 — Compare multiple models side by side
from huggingface_hub import model_info

models_to_compare = [
    "distilbert-base-uncased",
    "bert-base-uncased",
    "roberta-base",
]

print(f"{'Model':<35} {'Task':<25} {'Downloads':>12} {'Likes':>8}")
print("-" * 85)

for mid in models_to_compare:
    try:
        info = model_info(mid)
        downloads = getattr(info, 'downloads', 0) or 0
        print(f"{info.modelId:<35} {str(info.pipeline_tag):<25} {downloads:>12,} {info.likes:>8,}")
    except Exception as e:
        print(f"{mid:<35} Error: {e}")

---
## Task 3 — Load & Run Inference with Pre-trained Models (Transformers)

### The `transformers` library

Hugging Face `transformers` is the standard library for working with pre-trained models. It provides:

- **`pipeline()`** — the easiest high-level API. One line to load and run inference.
- **`AutoTokenizer`** — loads the correct tokenizer for any model automatically.
- **`AutoModelFor*`** — loads model weights for a specific task.

### The inference flow

```
Raw text
   ↓  Tokenizer.encode()
Token IDs (integers)
   ↓  Model.forward()
Logits / Hidden states
   ↓  Decoding strategy (greedy / beam / sampling)
Generated token IDs
   ↓  Tokenizer.decode()
Output text
```

### Installation

```bash
pip install transformers torch
# For GPU support:
pip install transformers torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
```

In [ ]:
# Cell 3.1 — Install transformers and torch
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install",
                "transformers", "torch", "accelerate", "-q"])

In [ ]:
# Cell 3.2 — Check device availability (CPU / GPU / MPS)
import torch

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"       # Apple Silicon
    print("Apple MPS (Metal) available")
else:
    device = "cpu"
    print("Running on CPU")

print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Cell 3.3 — Text generation with pipeline (easiest approach)
# 'gpt2' is tiny (124M params) — downloads in seconds.
# For better quality use 'gpt2-medium', 'gpt2-large', or 'distilgpt2'.

from transformers import pipeline

print("Loading model... (first run downloads weights)")

generator = pipeline(
    "text-generation",
    model="gpt2",
    device=0 if device == "cuda" else -1   # -1 = CPU
)

output = generator(
    "Artificial intelligence will change the world by",
    max_new_tokens=60,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.8,
    pad_token_id=generator.tokenizer.eos_token_id
)

print("Generated text:")
print(output[0]["generated_text"])

In [ ]:
# Cell 3.4 — Manual tokenization (understanding what happens under the hood)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Machine learning is fascinating."

# Step 1: encode text to token IDs
token_ids = tokenizer.encode(text)
print(f"Token IDs  : {token_ids}")

# Step 2: see what each token looks like
tokens = tokenizer.convert_ids_to_tokens(token_ids)
print(f"Tokens     : {tokens}")

# Step 3: decode back to text
decoded = tokenizer.decode(token_ids)
print(f"Decoded    : {decoded}")

print(f"\nVocab size : {tokenizer.vocab_size:,} tokens")
print(f"Token count: {len(token_ids)} tokens for {len(text.split())} words")

In [ ]:
# Cell 3.5 — Manual model forward pass (full control)
# This is what pipeline() does internally.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()   # disable dropout for deterministic output

prompt = "The future of artificial intelligence is"

# Tokenize — returns a dict with 'input_ids' and 'attention_mask'
inputs = tokenizer(prompt, return_tensors="pt")
print(f"Input IDs shape: {inputs['input_ids'].shape}")

# Generate
with torch.no_grad():    # no gradient computation needed for inference
    output_ids = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode only the NEW tokens (skip the prompt)
new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
generated  = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(f"\nPrompt    : {prompt}")
print(f"Completion: {generated}")

In [ ]:
# Cell 3.6 — Other pipeline tasks (not just text generation)
from transformers import pipeline

# --- Task A: Sentiment Analysis ---
print("=== Sentiment Analysis ===")
classifier = pipeline("sentiment-analysis",
                       model="distilbert-base-uncased-finetuned-sst-2-english",
                       device=-1)
reviews = [
    "This product is absolutely amazing!",
    "Terrible experience, would not recommend.",
    "It's okay, nothing special."
]
results = classifier(reviews)
for rev, res in zip(reviews, results):
    print(f"  [{res['label']:8s} {res['score']:.2f}]  {rev}")

# --- Task B: Fill-Mask ---
print("\n=== Fill-Mask (BERT) ===")
fill = pipeline("fill-mask", model="bert-base-uncased", device=-1)
result = fill("Paris is the [MASK] of France.")
for r in result[:3]:
    print(f"  {r['token_str']:<15} score={r['score']:.4f}  → {r['sequence']}")

# --- Task C: Summarization ---
print("\n=== Summarization ===")
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-6-6", device=-1)
text = (
    "Machine learning is a branch of artificial intelligence that gives computers "
    "the ability to learn without being explicitly programmed. It focuses on "
    "developing algorithms that can access data and use it to learn for themselves. "
    "Deep learning, a subset of machine learning, uses neural networks with many "
    "layers to model and process complex patterns in datasets."
)
summary = summarizer(text, max_length=50, min_length=20, do_sample=False)
print(f"  Original ({len(text.split())} words): {text[:80]}...")
print(f"  Summary  : {summary[0]['summary_text']}")

---
## Task 4 — Generation Parameters: temperature, top_p, top_k, max_tokens

### Why do generation parameters matter?

At every step, the model outputs a **probability distribution** over its entire vocabulary (~50,000 tokens for GPT-2). Generation parameters control *how you sample from that distribution* — they are the knobs that trade off **creativity vs. consistency**.

### Parameter reference

| Parameter | Type | Range | Effect |
|-----------|------|-------|--------|
| `temperature` | float | 0.0 – 2.0 | Controls randomness. Lower = more deterministic. 0 = always pick top token (greedy). |
| `top_k` | int | 1 – vocab_size | At each step, only consider the top-K most likely tokens. Low K = conservative. |
| `top_p` | float | 0.0 – 1.0 | Nucleus sampling: pick from smallest set of tokens whose cumulative prob ≥ p. |
| `max_new_tokens` | int | 1 – context_length | Maximum number of new tokens to generate. |
| `repetition_penalty` | float | 1.0 – 2.0 | Penalizes repeating the same tokens. 1.0 = no penalty. |
| `num_beams` | int | 1 – N | Beam search width. 1 = greedy/sampling. Higher = more thorough (slower). |
| `do_sample` | bool | True/False | Must be True to use temperature/top_p/top_k. False = greedy/beam. |

### Mental model for temperature

```
temperature = 0.0  →  Always picks highest-probability token  (deterministic, repetitive)
temperature = 0.7  →  Creative but coherent                   (good for chat, stories)
temperature = 1.0  →  Raw model probabilities unchanged       (baseline)
temperature = 1.5  →  More random, may be incoherent          (experimental)
temperature = 2.0  →  Nearly random token selection           (usually gibberish)
```

In [ ]:
# Cell 4.1 — Effect of temperature on output
# Run the same prompt with different temperatures and observe the difference.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model     = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

def generate_with_params(prompt, **kwargs):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            **kwargs
        )
    new = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True)


prompt = "The scientist discovered"
temperatures = [0.1, 0.7, 1.0, 1.5]

print(f"Prompt: \"{prompt}\"")
print("=" * 70)

for temp in temperatures:
    text = generate_with_params(
        prompt,
        max_new_tokens=40,
        do_sample=True,
        temperature=temp
    )
    label = "(conservative)" if temp <= 0.3 else "(creative)" if temp >= 1.3 else ""
    print(f"\ntemp={temp} {label}")
    print(f"  {text}")

In [ ]:
# Cell 4.2 — Effect of top_k and top_p

prompt = "In the year 2050, robots will"

configs = [
    {"label": "Greedy (no sampling)",     "do_sample": False},
    {"label": "top_k=5  (very focused)",  "do_sample": True, "top_k": 5,   "temperature": 0.8},
    {"label": "top_k=50 (moderate)",      "do_sample": True, "top_k": 50,  "temperature": 0.8},
    {"label": "top_p=0.5 (nucleus, tight)","do_sample": True, "top_p": 0.5, "temperature": 0.8},
    {"label": "top_p=0.95 (nucleus, wide)","do_sample": True, "top_p": 0.95,"temperature": 0.8},
    {"label": "top_k=50 + top_p=0.9",     "do_sample": True, "top_k": 50, "top_p": 0.9, "temperature": 0.8},
]

print(f"Prompt: \"{prompt}\"")
print("=" * 70)

for cfg in configs:
    label = cfg.pop("label")
    text  = generate_with_params(prompt, max_new_tokens=40, **cfg)
    print(f"\n[{label}]")
    print(f"  {text}")

In [ ]:
# Cell 4.3 — Repetition penalty and max_new_tokens

prompt = "The best way to learn programming is"

print(f"Prompt: \"{prompt}\"")
print("=" * 70)

# Without penalty — model may loop
text_no_penalty = generate_with_params(
    prompt, max_new_tokens=60, do_sample=True,
    temperature=0.7, repetition_penalty=1.0
)
print("\nrepetition_penalty=1.0 (none):")
print(" ", text_no_penalty)

# With penalty
text_with_penalty = generate_with_params(
    prompt, max_new_tokens=60, do_sample=True,
    temperature=0.7, repetition_penalty=1.3
)
print("\nrepetition_penalty=1.3:")
print(" ", text_with_penalty)

# max_new_tokens effect
print("\n--- max_new_tokens effect ---")
for n in [10, 30, 80]:
    text = generate_with_params(
        prompt, max_new_tokens=n, do_sample=True, temperature=0.8
    )
    print(f"max_new_tokens={n:<3}: {text[:100]}{'...' if len(text)>100 else ''}")

In [ ]:
# Cell 4.4 — Same parameters via Ollama API
# Ollama accepts generation parameters in the 'options' dict.

import requests

OLLAMA_BASE = "http://localhost:11434"

def ollama_with_params(prompt, model="llama3.1", **options):
    payload = {"model": model, "prompt": prompt, "stream": False, "options": options}
    r = requests.post(f"{OLLAMA_BASE}/api/generate", json=payload)
    return r.json()["response"]


prompt = "Describe how neural networks learn in simple terms:"

configs = [
    {"label": "Low temp  (focused)",   "temperature": 0.2, "top_p": 0.9},
    {"label": "Med temp  (balanced)",  "temperature": 0.7, "top_p": 0.9},
    {"label": "High temp (creative)",  "temperature": 1.4, "top_p": 0.95},
]

print(f"Prompt: {prompt}")
print("=" * 70)

for cfg in configs:
    label = cfg.pop("label")
    try:
        result = ollama_with_params(prompt, num_predict=80, **cfg)
        print(f"\n[{label}]\n  {result[:200]}")
    except Exception as e:
        print(f"\n[{label}] Ollama not available: {e}")

In [ ]:
# Cell 4.5 — Parameter cheat-sheet for different use cases

use_cases = [
    {"Use case": "Factual Q&A / Code",    "temperature": 0.1, "top_k": 10,  "top_p": 0.9,  "rep_penalty": 1.0},
    {"Use case": "Chat assistant",         "temperature": 0.7, "top_k": 50,  "top_p": 0.9,  "rep_penalty": 1.1},
    {"Use case": "Creative writing",       "temperature": 1.0, "top_k": 100, "top_p": 0.95, "rep_penalty": 1.2},
    {"Use case": "Poetry / experimental",  "temperature": 1.4, "top_k": 0,   "top_p": 0.95, "rep_penalty": 1.3},
    {"Use case": "Deterministic output",   "temperature": 0.0, "top_k": 1,   "top_p": 1.0,  "rep_penalty": 1.0},
]

header = f"{'Use case':<25} {'temp':>6} {'top_k':>6} {'top_p':>6} {'rep_pen':>8}"
print(header)
print("-" * len(header))
for u in use_cases:
    print(f"{u['Use case']:<25} {u['temperature']:>6} {u['top_k']:>6} {u['top_p']:>6} {u['rep_penalty']:>8}")

---
## Task 5 — CLI Tool for Model Inference Comparison

### What we're building

A command-line interface (CLI) tool that:
- Accepts a **prompt** from the user
- Runs it against **multiple models** (Ollama + Hugging Face)
- Displays **side-by-side comparison** of outputs
- Measures and shows **latency** for each model
- Accepts **generation parameters** as CLI flags

### Usage (once saved as a .py file)

```bash
# Basic usage
python llm_compare.py "What is recursion?"

# With parameters
python llm_compare.py "Write a haiku about Python" --temperature 1.2 --max_tokens 60

# Specify which models to compare
python llm_compare.py "Explain list comprehension" --models ollama gpt2

# Save output to file
python llm_compare.py "What is OOP?" --output result.txt
```

The next two cells define the tool — first the code, then we run it directly from the notebook.

In [ ]:
# Cell 5.1 — Write the CLI tool to disk
# We use %%writefile magic to save this as a standalone Python script.

cli_code = '''
#!/usr/bin/env python3
"""
llm_compare.py — CLI tool to compare inference across multiple models.

Usage:
    python llm_compare.py "Your prompt here" [options]

Options:
    --models       Space-separated list: ollama gpt2 distilgpt2 (default: all)
    --temperature  Float 0.0-2.0  (default: 0.7)
    --top_p        Float 0.0-1.0  (default: 0.9)
    --top_k        Int            (default: 50)
    --max_tokens   Int            (default: 80)
    --ollama_model Ollama model name (default: llama3.1)
    --output       Path to save results (optional)
"""

import argparse
import time
import sys
import textwrap


# ─── Ollama backend ───────────────────────────────────────────────────────────

def run_ollama(prompt, model="llama3.1", temperature=0.7,
               top_p=0.9, top_k=50, max_tokens=80):
    try:
        import requests
        payload = {
            "model":  model,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": temperature,
                "top_p":       top_p,
                "top_k":       top_k,
                "num_predict": max_tokens,
            }
        }
        r = requests.post("http://localhost:11434/api/generate",
                          json=payload, timeout=120)
        r.raise_for_status()
        return r.json()["response"].strip()
    except Exception as e:
        return f"[Ollama error: {e}]"


# ─── Hugging Face backend ─────────────────────────────────────────────────────

def run_hf_model(prompt, model_name="gpt2", temperature=0.7,
                 top_p=0.9, top_k=50, max_tokens=80):
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM

        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model     = AutoModelForCausalLM.from_pretrained(model_name)
        model.eval()

        inputs = tokenizer(prompt, return_tensors="pt")
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1,
            )
        new_ids = out[0][inputs["input_ids"].shape[1]:]
        return tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    except Exception as e:
        return f"[HF error: {e}]"


# ─── Dispatcher ───────────────────────────────────────────────────────────────

BACKENDS = {
    "ollama":     ("Ollama (llama3.1)",  lambda p, a: run_ollama(p, a.ollama_model, a.temperature, a.top_p, a.top_k, a.max_tokens)),
    "gpt2":       ("GPT-2 (124M)",       lambda p, a: run_hf_model(p, "gpt2",       a.temperature, a.top_p, a.top_k, a.max_tokens)),
    "distilgpt2": ("DistilGPT-2 (82M)", lambda p, a: run_hf_model(p, "distilgpt2", a.temperature, a.top_p, a.top_k, a.max_tokens)),
}


# ─── Display helpers ──────────────────────────────────────────────────────────

def box(title, text, width=72):
    border = "─" * (width - 2)
    lines  = textwrap.wrap(text, width - 4) if text else ["(no output)"]
    body   = "\n".join(f"│  {l:<{width-4}}  │" for l in lines)
    return f"┌{border}┐\n│  {title:<{width-4}}  │\n├{border}┤\n{body}\n└{border}┘"


# ─── Main ─────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(
        description="Compare LLM outputs across multiple models."
    )
    parser.add_argument("prompt",        type=str,   help="Input prompt")
    parser.add_argument("--models",      nargs="+",  default=list(BACKENDS.keys()),
                        choices=list(BACKENDS.keys()), help="Models to compare")
    parser.add_argument("--temperature", type=float, default=0.7)
    parser.add_argument("--top_p",       type=float, default=0.9)
    parser.add_argument("--top_k",       type=int,   default=50)
    parser.add_argument("--max_tokens",  type=int,   default=80)
    parser.add_argument("--ollama_model",type=str,   default="llama3.1")
    parser.add_argument("--output",      type=str,   default=None)
    args = parser.parse_args()

    separator = "═" * 72
    output_lines = []

    header = [
        separator,
        " LLM INFERENCE COMPARISON",
        separator,
        f" Prompt      : {args.prompt}",
        f" Models      : {args.models}",
        f" temperature={args.temperature}  top_p={args.top_p}  top_k={args.top_k}  max_tokens={args.max_tokens}",
        separator,
    ]
    for line in header:
        print(line)
        output_lines.append(line)

    results_summary = []

    for key in args.models:
        display_name, runner = BACKENDS[key]
        print(f"\nRunning {display_name}...", end=" ", flush=True)

        t0 = time.time()
        result = runner(args.prompt, args)
        elapsed = time.time() - t0

        print(f"done ({elapsed:.1f}s)")

        title = f"{display_name}  [{elapsed:.1f}s]"
        b = box(title, result)
        print(b)
        output_lines.append(b)
        results_summary.append((display_name, elapsed, len(result.split())))

    # Summary table
    print(f"\n{separator}")
    print(" SUMMARY")
    print(separator)
    print(f" {'Model':<25} {'Latency':>10} {'Words':>8}")
    print(f" {\'─\'*25} {\'─\'*10} {\'─\'*8}")
    for name, lat, words in results_summary:
        print(f" {name:<25} {lat:>9.1f}s {words:>8}")
    print(separator)

    if args.output:
        with open(args.output, "w") as f:
            f.write("\n".join(output_lines))
        print(f"\nResults saved to: {args.output}")


if __name__ == "__main__":
    main()
'''

with open("llm_compare.py", "w") as f:
    f.write(cli_code)

print("llm_compare.py written successfully!")
print("\nUsage examples:")
print("  python llm_compare.py \"What is recursion?\"")
print("  python llm_compare.py \"Write a haiku\" --temperature 1.2 --max_tokens 40")
print("  python llm_compare.py \"Explain OOP\" --models gpt2 distilgpt2")

In [ ]:
# Cell 5.2 — Run the comparison tool directly from the notebook
# This simulates what happens when you call the CLI tool.

import time
import textwrap
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def run_hf_model_nb(prompt, model_name, temperature=0.7, top_p=0.9, top_k=50, max_tokens=80):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForCausalLM.from_pretrained(model_name)
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
    new_ids = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

def print_box(title, text, width=70):
    border = "─" * (width - 2)
    lines  = textwrap.wrap(text, width - 6) or ["(no output)"]
    print(f"┌{border}┐")
    print(f"│  {title:<{width-4}}  │")
    print(f"├{border}┤")
    for line in lines:
        print(f"│  {line:<{width-4}}  │")
    print(f"└{border}┘")

# ── Configuration ───────────────────────────────────────────────
PROMPT      = "Explain what machine learning is in simple terms."
TEMPERATURE = 0.7
TOP_P       = 0.9
TOP_K       = 50
MAX_TOKENS  = 60
MODELS      = [("GPT-2", "gpt2"), ("DistilGPT-2", "distilgpt2")]
# ─────────────────────────────────────────────────────────────────

print("═" * 70)
print(" LLM INFERENCE COMPARISON (notebook run)")
print("═" * 70)
print(f" Prompt      : {PROMPT}")
print(f" temperature={TEMPERATURE}  top_p={TOP_P}  top_k={TOP_K}  max_tokens={MAX_TOKENS}")
print("═" * 70)

summary = []
for name, model_id in MODELS:
    print(f"\nRunning {name}...", end=" ", flush=True)
    t0 = time.time()
    result = run_hf_model_nb(PROMPT, model_id, TEMPERATURE, TOP_P, TOP_K, MAX_TOKENS)
    elapsed = time.time() - t0
    print(f"done ({elapsed:.1f}s)")
    print_box(f"{name}  [{elapsed:.1f}s]", result)
    summary.append((name, elapsed, len(result.split())))

print("\n" + "═" * 70)
print(" SUMMARY")
print("═" * 70)
print(f" {'Model':<20} {'Latency':>10} {'Words out':>10}")
print(f" {'─'*20} {'─'*10} {'─'*10}")
for name, lat, words in summary:
    print(f" {name:<20} {lat:>9.1f}s {words:>10}")
print("═" * 70)

In [ ]:
# Cell 5.3 — Run the CLI tool via terminal (uses ! to run shell commands)
# Make sure llm_compare.py was written by Cell 5.1 first.

# Compare gpt2 and distilgpt2 only (Ollama must be running for ollama backend)
!python llm_compare.py "What is object-oriented programming?" \
    --models gpt2 distilgpt2 \
    --temperature 0.7 \
    --max_tokens 60

In [ ]:
# Cell 5.4 — Save comparison results to a file

!python llm_compare.py "Explain recursion in programming." \
    --models gpt2 distilgpt2 \
    --temperature 0.8 \
    --max_tokens 80 \
    --output comparison_results.txt

print("\n--- Contents of comparison_results.txt ---")
with open("comparison_results.txt") as f:
    print(f.read())

---
## Summary & Key Takeaways

### What you implemented

| Task | What you did | Tools used |
|------|-------------|------------|
| 1 | Installed Ollama, pulled Llama 3.1 8B, called generate & chat APIs | `ollama`, `requests` |
| 2 | Searched HF Hub, read model cards, compared models by downloads/license | `huggingface_hub` |
| 3 | Loaded GPT-2, ran pipeline and manual inference, explored other tasks | `transformers`, `torch` |
| 4 | Systematically tested temperature, top_k, top_p, repetition_penalty | `model.generate()` kwargs |
| 5 | Built a CLI tool with argparse to compare multiple models side-by-side | `argparse`, all backends |

### Mental models to keep

- **Ollama** = local model server. Ideal for running large open-source models without cloud costs.
- **Hugging Face** = the library + the hub. `transformers` loads models; `huggingface_hub` explores them.
- **Temperature** controls creativity; **top_k/top_p** control the candidate pool; **max_tokens** controls length.
- **`do_sample=False`** = greedy/beam (deterministic). **`do_sample=True`** = stochastic (uses temp/top_k/top_p).
- Always check the **license** before using a model commercially.

### Good default settings for most use cases

```python
temperature    = 0.7
top_p          = 0.9
top_k          = 50
max_new_tokens = 200
repetition_penalty = 1.1
do_sample      = True
```

### Next steps

- Fine-tune a model on a custom dataset using `transformers.Trainer`
- Try quantized models (4-bit via `bitsandbytes`) for memory efficiency
- Explore function calling / tool use with Ollama's chat API
- Build a RAG (Retrieval-Augmented Generation) pipeline